# Wave-packet scattering from a potential step and a finite barrier

## Numerical solution of the time-dependent Schrödinger equation

In textbook calculations, scattering is usually described with stationary plane waves. A real particle is better represented by a **wave packet**: it is localized, contains a range of momenta and visibly separates into reflected and transmitted parts.

This notebook solves

$$
i\hbar\frac{\partial\Psi(x,t)}{\partial t}
=\left[-\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2}+V(x)\right]\Psi(x,t)
$$

numerically for an electron incident on:

1. a potential step;
2. a finite rectangular barrier.

You do **not** need to write code. Choose the parameters, click **Run simulation**, and interpret the animation using the explanations below.

### What will be animated?

- a snapshot-style main panel in which the moving $|\Psi(x,t)|^2$ packet is drawn together with the actual step or barrier profile;
- $\operatorname{Re}\Psi(x,t)$ and $\operatorname{Im}\Psi(x,t)$ in a separate panel;
- the probabilities to the left, inside/near the potential, and to the right;
- the separation into reflected and transmitted packets.

In the main panel, the density is multiplied by a constant visual scale so that it can be seen beside $V(x)$. The black potential retains its value in eV; the rescaled purple height is not an energy. The shape, position and integrated area of the density—not its displayed height relative to $V_0$—carry the physical information.

### Recommended working method

Change **one parameter at a time**. Begin with the suggested cases in each section, predict the outcome, run the simulation, and only then open the hidden explanation.

## 1. From a stationary wave to a moving packet

The initial state is a Gaussian packet,

$$
\Psi(x,0)=A\exp\left[-\frac{(x-x_0)^2}{4\sigma_x^2}\right]e^{ik_0x},
$$

whose carrier wave number $k_0$ defines the **central (carrier) energy**

$$E_0=\frac{\hbar^2k_0^2}{2m}.$$

The Gaussian envelope localizes the particle near $x_0$, while $e^{ik_0x}$ makes it move to the right. The density $|\Psi|^2$ has standard deviation $\sigma_x$.

### The Gaussian packet is a superposition of plane waves

A plane wave $e^{ikx}$ extends over all space and has a perfectly defined momentum $p=\hbar k$. It cannot represent a localized particle. A localized packet is constructed by superposing plane waves with different values of $k$:

$$
\Psi(x,0)=\frac{1}{\sqrt{2\pi}}
\int_{-\infty}^{\infty}\widetilde\Psi(k)e^{ikx}\,dk.
$$

This is a continuous, and therefore idealized **infinite superposition of plane waves**. The coefficient $\widetilde\Psi(k)$ gives the complex amplitude of each plane-wave component.

For the Gaussian packet used here, its Fourier transform is also Gaussian:

$$
\widetilde\Psi(k)\propto
\exp\left[-\sigma_x^2(k-k_0)^2\right]
e^{-i(k-k_0)x_0}.
$$

Therefore, the packet does not contain all wave numbers with equal importance. Components near $k_0$ dominate, while components far from $k_0$ have exponentially small amplitudes. The phase factor involving $x_0$ places their constructive interference around $x=x_0$.

This gives a useful physical picture:

- near $x_0$, the plane-wave components interfere mostly constructively;
- far from $x_0$, their different phases cause mostly destructive interference;
- the result is a localized Gaussian envelope rather than an extended plane wave.

### Position and momentum uncertainty

Localization requires a spread of wave numbers. Approximately,

$$
\Delta x=\sigma_x,
\qquad
\Delta k=\frac{1}{2\sigma_x},
\qquad
\Delta p=\hbar\Delta k.
$$

Consequently,

$$
\Delta x\,\Delta p=\frac{\hbar}{2}.
$$

The Gaussian packet is a minimum-uncertainty state. Changing $\sigma_x$ illustrates the reciprocal relation between localization and momentum spread:

- **large $\sigma_x$:** broad in position, narrow in $k$ and closer to a monochromatic plane wave;
- **small $\sigma_x$:** well localized, but broad in momentum and energy.

### A wave packet never has one exact energy

Each plane-wave component has its own free-particle energy,

$$
E(k)=\frac{\hbar^2k^2}{2m}.
$$

The quantity controlled in this notebook is $E_0=\hbar^2k_0^2/(2m)$, the energy associated with the central wave number. It is **not an exact energy of the whole packet**. For this Gaussian,

$$
\langle E\rangle
=\frac{\hbar^2}{2m}\left(k_0^2+\Delta k^2\right)
=E_0+\frac{\hbar^2}{8m\sigma_x^2}.
$$

Thus, the expectation value is slightly greater than the carrier energy, and the state contains a distribution of energies on both sides of it.

This matters in the scattering simulations. Every $k$ component has its own reflection and transmission coefficients, $R(k)$ and $T(k)$. The outgoing reflected and transmitted packets are the sums of all those differently scattered components. Therefore:

- numerical $R$ and $T$ need not exactly equal the plane-wave result evaluated at $E_0$;
- a packet with $E_0<V_0$ may contain a small high-energy tail with $E(k)>V_0$;
- sharp transmission resonances can be blurred when the packet has a broad energy distribution.

### Why does the packet move and spread?

For a free particle, every plane-wave component evolves as

$$
e^{ikx}\longrightarrow e^{ikx}e^{-iE(k)t/\hbar}.
$$

Because $E(k)\propto k^2$, different components accumulate phase at different rates. Their interference maximum moves with the group velocity, while the phases gradually lose their original alignment and the packet spreads. During scattering, the potential additionally changes the amplitude and phase of each component.

<details><summary><b>How can an infinite superposition be represented numerically?</b></summary>

Mathematically, the Fourier representation is a continuous integral over infinitely many $k$ values. The computer stores $\Psi$ at a finite number of spatial grid points. In this notebook, $N=2048$, so the Fast Fourier Transform represents it using 2048 discrete Fourier components:

$$
\Psi(x_j)\quad\longleftrightarrow\quad\widetilde\Psi(k_n).
$$

This finite sum approximates the continuous integral. The spatial interval determines the spacing between allowed $k_n$, while the grid spacing determines the largest wave number that can be represented. The split-step method evolves all these discrete plane-wave components simultaneously.
</details>

<details><summary><b>Why plot both the wavefunction and its density?</b></summary>

$\Psi$ contains phase and sign information, which control interference. It is complex, so the animation displays its real and imaginary parts. The observable position distribution is $|\Psi|^2$, which is real and non-negative. Oscillations of $\operatorname{Re}\Psi$ do not mean that probability alternates between positive and negative values.
</details>

## 2. Numerical method: split-step Fourier propagation

For a short time interval $\Delta t$, the evolution operator is approximated by

$$
e^{-iH\Delta t/\hbar}
\approx
e^{-iV\Delta t/(2\hbar)}
e^{-iT\Delta t/\hbar}
e^{-iV\Delta t/(2\hbar)},
$$

where $T=p^2/(2m)$.

The potential part is easiest to apply in position space. The kinetic part is easiest in momentum space, where it is simply multiplication by $e^{-i\hbar k^2\Delta t/(2m)}$. A Fast Fourier Transform repeatedly moves the wavefunction between the two representations.

The method preserves the norm extremely well. It also solves the full time-dependent problem: reflection, transmission, interference and temporary probability inside the barrier emerge from the Schrödinger equation rather than being inserted by hand.

> The discontinuous potentials are idealizations. The numerical spatial grid represents their edges with finite resolution.

In [ ]:
# Run this setup cell once
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.integrate import simpson
from matplotlib.animation import FuncAnimation
from IPython.display import display, HTML, Markdown, clear_output

plt.rcParams.update({"font.size": 10, "figure.figsize": (10, 8)})

# Units: energy in eV, distance in Å, time in fs
HBAR = 0.6582119569                 # eV fs
C_ELECTRON = 3.809982116           # ħ²/(2m_e), eV Å²

def plane_wave_transmission(kind, E, V0, width=0.0):
    """Analytical transmission coefficient for a monochromatic plane wave."""
    if kind == 'step':
        if E <= V0:
            return 0.0
        k1 = np.sqrt(E/C_ELECTRON)
        k2 = np.sqrt((E-V0)/C_ELECTRON)
        return 4*k1*k2/(k1+k2)**2

    # Rectangular barrier
    if np.isclose(E, V0, rtol=1e-9, atol=1e-12):
        return 1.0/(1.0 + V0*width**2/(4*C_ELECTRON))
    if E < V0:
        kappa = np.sqrt((V0-E)/C_ELECTRON)
        return 1.0/(1.0 + V0**2*np.sinh(kappa*width)**2/(4*E*(V0-E)))
    q = np.sqrt((E-V0)/C_ELECTRON)
    return 1.0/(1.0 + V0**2*np.sin(q*width)**2/(4*E*(E-V0)))

def simulate_wavepacket(kind='step', V0=0.6, energy_ratio=1.5,
                        width=8.0, sigma=10.0, n_frames=72):
    """Propagate a Gaussian electron wave packet and return animation data."""
    E = V0*energy_ratio
    N = 2048
    x = np.linspace(-140.0, 140.0, N, endpoint=False)
    dx = x[1]-x[0]
    x0 = -70.0
    k0 = np.sqrt(E/C_ELECTRON)
    delta_k = 1.0/(2.0*sigma)
    E_expectation = C_ELECTRON*(k0**2 + delta_k**2)

    psi = np.exp(-(x-x0)**2/(4*sigma**2))*np.exp(1j*k0*x)
    psi /= np.sqrt(np.sum(np.abs(psi)**2)*dx)

    if kind == 'step':
        V = np.where(x >= 0.0, V0, 0.0)
        left_edge, right_edge = 0.0, 0.0
        buffer = 5.0
    else:
        V = np.where(np.abs(x) <= width/2, V0, 0.0)
        left_edge, right_edge = -width/2, width/2
        buffer = 4.0

    # Mean incident group velocity and a duration long enough to see separation
    velocity = 2*C_ELECTRON*k0/HBAR  # Å/fs
    t_final = (abs(x0-left_edge) + 55.0)/velocity
    target_dt = 0.025
    steps_per_frame = max(1, int(np.ceil(t_final/((n_frames-1)*target_dt))))
    dt = t_final/((n_frames-1)*steps_per_frame)

    k = 2*np.pi*np.fft.fftfreq(N, d=dx)
    phase_V_half = np.exp(-0.5j*V*dt/HBAR)
    phase_T = np.exp(-1j*C_ELECTRON*k**2*dt/HBAR)

    frames = np.empty((n_frames, N), dtype=np.complex128)
    frames[0] = psi
    for frame in range(1, n_frames):
        for _ in range(steps_per_frame):
            psi *= phase_V_half
            psi = np.fft.ifft(phase_T*np.fft.fft(psi))
            psi *= phase_V_half
        frames[frame] = psi

    times = np.linspace(0, t_final, n_frames)
    density = np.abs(frames)**2
    left_mask = x < left_edge-buffer
    right_mask = x > right_edge+buffer
    middle_mask = ~(left_mask | right_mask)
    P_left = density[:, left_mask].sum(axis=1)*dx
    P_right = density[:, right_mask].sum(axis=1)*dx
    P_middle = density[:, middle_mask].sum(axis=1)*dx

    result = {
        'x':x, 'V':V, 'frames':frames, 'density':density, 'times':times,
        'P_left':P_left, 'P_right':P_right, 'P_middle':P_middle,
        'E':E, 'E_expectation':E_expectation, 'V0':V0, 'width':width, 'kind':kind, 'dx':dx,
        'T_plane':plane_wave_transmission(kind,E,V0,width),
        'norm':density[-1].sum()*dx, 'sigma':sigma
    }
    return result

def build_animation(result):
    x=result['x']; V=result['V']; frames=result['frames']; dens=result['density']
    times=result['times']; PL=result['P_left']; PR=result['P_right']; PM=result['P_middle']
    # The first two panels use position; the third uses time. They must NOT share x axes.
    fig, axes = plt.subplots(3, 1, figsize=(10, 9.5), constrained_layout=True,
                             gridspec_kw={'height_ratios':[1.20,0.92,0.72]})
    axd, axw, axp = axes
    amp_max = 1.12*np.max(np.abs(frames))
    # Scale density only for the snapshot-style overlay. V(x) itself remains in eV.
    density_scale = 0.72*result['V0']/max(dens[0].max(),1e-15)
    dens_display = density_scale*dens
    display_ymax = max(1.28*result['V0'],1.08*np.max(dens_display),1.08*result['E'])

    # Exact potential outline, visibly drawn as in the reference snapshots
    potential_line, = axd.plot(x,V,color='black',lw=2.8,label='$V(x)$')
    energy_line=axd.axhline(result['E'],color='#059669',ls='--',lw=1.3,
                            label='carrier energy $E_0$')
    line_den, = axd.plot([],[],color='#6d28d9',lw=2.2,label='scaled $|\\Psi|^2$')
    fill_holder=[None]
    line_re, = axw.plot([],[],color='#2563eb',lw=1.5,label='$\\mathrm{Re}\\,\\Psi$')
    line_im, = axw.plot([],[],color='#f97316',lw=1.2,label='$\\mathrm{Im}\\,\\Psi$')
    line_L, = axp.plot([],[],color='#2563eb',lw=2,label='$P_{left}$')
    line_M, = axp.plot([],[],color='#64748b',lw=2,label='$P_{interaction}$')
    line_R, = axp.plot([],[],color='#f97316',lw=2,label='$P_{right}$')
    time_text=axd.text(0.82,0.88,'',transform=axd.transAxes,
                       bbox=dict(facecolor='white',alpha=.8,edgecolor='none'))

    # Keep the spatial window fixed while the probability panel advances in time.
    axd.set_xlim(-90,90); axw.set_xlim(-90,90); axp.set_xlim(0,times[-1])
    axd.set_ylim(0,display_ymax); axw.set_ylim(-amp_max,amp_max); axp.set_ylim(0,1.03)
    axd.set_ylabel('Potential $V$ (eV)\nand scaled density')
    axw.set_ylabel('Wavefunction\n(arbitrary amplitude)')
    axw.set_xlabel('Position $x$ (Å)')
    axp.set_ylabel('Integrated\nprobability'); axp.set_xlabel('Time $t$ (fs)')
    axd.set_title(f"{result['kind'].capitalize()} scattering:  E/V₀={result['E']/result['V0']:.2f},  V₀={result['V0']:.2f} eV")
    for ax in axes: ax.grid(alpha=.18)
    axd.legend(loc='upper left',ncol=3); axw.legend(loc='upper left'); axp.legend(loc='center left')

    def update(i):
        line_re.set_data(x,frames[i].real); line_im.set_data(x,frames[i].imag)
        line_den.set_data(x,dens_display[i])
        if fill_holder[0] is not None: fill_holder[0].remove()
        fill_holder[0]=axd.fill_between(x,0,dens_display[i],color='#8b5cf6',alpha=.30)
        line_L.set_data(times[:i+1],PL[:i+1]); line_M.set_data(times[:i+1],PM[:i+1]); line_R.set_data(times[:i+1],PR[:i+1])
        time_text.set_text(f"t = {times[i]:.2f} fs")
        return line_re,line_im,line_den,line_L,line_M,line_R,time_text

    anim=FuncAnimation(fig,update,frames=len(times),interval=65,blit=False)
    plt.close(fig)
    return anim

print('Numerical engine ready.')

## 3. Potential step

The potential is

$$
V(x)=\begin{cases}0,&x<0,\\V_0,&x\geq0.\end{cases}
$$

### Predictions before running

- **$E>V_0$:** both reflection and transmission occur. In the raised region the transmitted wavelength is longer because its kinetic energy is $E-V_0$.
- **$E<V_0$:** a monochromatic component with that energy has no propagating transmitted wave. It is reflected while an evanescent tail temporarily penetrates into $x>0$. A finite packet may still transmit a small high-energy component because it contains a range of energies.
- **$E\gg V_0$:** the mismatch becomes small and reflection decreases.

The shaded red half-plane marks the region in which $V=V_0$. The third panel uses spatial partitions only while the packet interacts. Once the pieces are separated, its final left and right values approximate $R$ and $T$.

### Suggested cases

1. $E/V_0=1.5$: partial reflection and transmission.
2. $E/V_0=0.7$: total reflection with penetration.
3. $E/V_0=1.02$: slow transmitted component near threshold.
4. $E/V_0=2.5$: weak reflection.

In [ ]:
def make_scattering_lab(kind):
    V0=widgets.FloatSlider(value=0.6,min=0.2,max=2.0,step=0.1,
                           description='$V_0$ (eV):',continuous_update=False)
    ratio=widgets.FloatSlider(value=1.5,min=0.35,max=2.5,step=0.05,
                              description='$E/V_0$:',continuous_update=False)
    sigma=widgets.FloatSlider(value=10.0,min=6.0,max=16.0,step=1.0,
                              description='$\\sigma_x$ (Å):',continuous_update=False)
    width=None
    controls=[V0,ratio]
    if kind=='barrier':
        width=widgets.FloatSlider(value=8.0,min=2.0,max=20.0,step=1.0,
                                  description='Width (Å):',continuous_update=False)
        controls.append(width)
    controls.append(sigma)
    run=widgets.Button(description='Run simulation',button_style='primary',icon='play')
    status=widgets.HTML(value='<i>Choose parameters and click Run simulation.</i>')
    out=widgets.Output()

    def on_run(_):
        run.disabled=True; status.value='<b>Computing the wave-packet evolution…</b>'
        with out:
            clear_output(wait=True)
            w=width.value if width is not None else 0.0
            result=simulate_wavepacket(kind,V0.value,ratio.value,w,sigma.value)
            Rnum=result['P_left'][-1]; Tnum=result['P_right'][-1]; M=result['P_middle'][-1]
            Tpw=result['T_plane']
            display(Markdown(
                f"**Carrier energy:** $E_0={result['E']:.3f}$ eV; "
                f"**energy expectation:** $\\langle E\\rangle={result['E_expectation']:.3f}$ eV  "
                f"($E/V_0={result['E']/result['V0']:.2f}$)  \\n"
                f"**Late-time spatial probabilities:** $P_{{left}}={Rnum:.3f}$, "
                f"$P_{{right}}={Tnum:.3f}$, $P_{{interaction}}={M:.3f}$  \\n"
                f"**Plane-wave prediction at the carrier energy $E_0$:** $R={1-Tpw:.3f}$, $T={Tpw:.3f}$  \\n"
                f"**Final numerical norm:** {result['norm']:.6f}"
            ))
            display(HTML(build_animation(result).to_jshtml()))
        status.value='<span style="color:green"><b>Simulation complete.</b></span>'
        run.disabled=False

    run.on_click(on_run)
    display(widgets.VBox(controls+[run,status]),out)
    return on_run

step_runner=make_scattering_lab('step')

### Questions about the step

**1. Why is there reflection even when $E>V_0$?**

<details><summary><b>Show explanation</b></summary>

The wave number changes abruptly from $k_1=\sqrt{2mE}/\hbar$ to $k_2=\sqrt{2m(E-V_0)}/\hbar$. Continuity of $\Psi$ and $d\Psi/dx$ at the boundary cannot be satisfied by a transmitted wave alone; a reflected component is required. Reflection is caused by wave-number mismatch, not only by classically forbidden motion.
</details>

**2. Why is the transmitted wavelength longer for $E>V_0$?**

<details><summary><b>Show explanation</b></summary>

The total energy is unchanged, but in the region $x>0$ the potential energy is $V_0$, leaving kinetic energy $E-V_0$. Therefore $k_2<k_1$ and $\lambda_2=2\pi/k_2$ is larger. The transmitted packet also moves more slowly.
</details>

**3. For $E<V_0$, does probability enter the forbidden region?**

<details><summary><b>Show explanation</b></summary>

Yes. A monochromatic component below the step penetrates as an evanescent tail, approximately $e^{-\kappa x}$ with $\kappa=\sqrt{2m(V_0-E)}/\hbar$, but carries no persistent transmitted current to infinity. A finite Gaussian packet also contains energies above its carrier energy. If some lie above $V_0$, the animation can show a small genuinely transmitted high-energy packet even when the displayed $E_0$ satisfies $E_0<V_0$.
</details>

**4. Why can the numerical result differ from the displayed plane-wave coefficient?**

<details><summary><b>Show explanation</b></summary>

The analytical value assumes one exact energy. The Gaussian packet contains an energy distribution, and each component has a different $R(E)$ and $T(E)$. The numerical result averages over that distribution. In addition, the reported spatial probabilities are taken at a finite time, so a small part may still lie near the interface.
</details>

## 4. Finite rectangular barrier

Now

$$
V(x)=\begin{cases}
V_0,&|x|\leq a/2,\\
0,&|x|>a/2,
\end{cases}
$$

where the widget labelled **Width** is $a$.

Unlike the step, the potential returns to zero after a finite distance. Therefore, even when $E<V_0$, an evanescent wave can reach the second edge and generate a transmitted packet: **quantum tunnelling**.

### What the width changes

For $E<V_0$, transmission decreases approximately as

$$T\propto e^{-2\kappa a},\qquad
\kappa=\frac{\sqrt{2m(V_0-E)}}{\hbar}.$$

For $E>V_0$, waves reflected at the two barrier edges interfere. Depending on the phase accumulated inside, increasing the width need not decrease transmission monotonically. At resonant widths the barrier can become highly transparent.

### Suggested cases

1. Fix $E/V_0=0.7$ and compare widths 4, 8 and 14 Å.
2. Fix the width at 8 Å and move $E/V_0$ from 0.5 toward 1.
3. Choose $E/V_0=1.5$ and scan the width: look for non-monotonic transmission.
4. Compare a narrow packet ($\sigma_x=6$ Å) with a broad packet ($16$ Å).

In [ ]:
barrier_runner=make_scattering_lab('barrier')

### Questions about the barrier

**1. How can a particle with $E<V_0$ appear beyond the barrier without gaining energy?**

<details><summary><b>Show explanation</b></summary>

Energy is conserved. Inside the barrier the stationary components are evanescent rather than propagating, but the wavefunction remains nonzero. If the barrier is finite, the evanescent amplitude can reach the second edge and connect to a propagating wave on the far side. No energy is borrowed and the transmitted packet has the same total-energy distribution as the incident packet.
</details>

**2. Why does barrier width affect tunnelling so strongly?**

<details><summary><b>Show explanation</b></summary>

The amplitude decays approximately as $e^{-\kappa x}$ inside a classically forbidden barrier, so the probability transmitted across width $a$ contains approximately $e^{-2\kappa a}$. Width therefore appears in an exponential, not merely as a linear correction.
</details>

**3. For $E>V_0$, why can transmission oscillate with barrier width?**

<details><summary><b>Show explanation</b></summary>

Inside the barrier the wave is propagating when $E>V_0$. Multiple partial reflections occur at both edges. Their transmitted contributions interfere. When the internal phase satisfies approximately $qa=n\pi$, destructive reflection or constructive transmission gives a resonance with $T\approx1$.
</details>

**4. Why are resonances less sharp for a narrow spatial packet?**

<details><summary><b>Show explanation</b></summary>

A narrow packet has a large momentum and energy spread. A sharp plane-wave resonance occurs only for a narrow range of energies, so different components of the packet experience different transmission. The averaged result smooths the resonance. A broader spatial packet is closer to a monochromatic plane wave.
</details>

**5. Is the oscillation of $\operatorname{Re}\Psi$ inside the barrier an oscillation in probability?**

<details><summary><b>Show explanation</b></summary>

No. The real and imaginary components depend on phase and can oscillate rapidly. Probability is described by $|\Psi|^2$. Spatial fringes in the density can occur when incident and reflected amplitudes overlap, but negative values of $\operatorname{Re}\Psi$ are not negative probabilities.
</details>

## 5. Connecting the animation with stationary scattering

| Feature | Stationary plane-wave treatment | Wave-packet animation |
|---|---|---|
| Incident state | Extends over all space | Initially localized |
| Energy | Exactly defined | Distribution centred on $E$ |
| Reflection/transmission | Constant flux ratios $R$ and $T$ | Integrated probabilities after packets separate |
| Interference | Spatial stationary pattern | Temporary fringes while components overlap |
| Time dependence | Only an overall $e^{-iEt/\hbar}$ for each stationary state | Packet motion and separation are visible |

The two descriptions are consistent. A packet is a superposition of stationary scattering states. If it is sufficiently broad in space and narrow in energy, its final numerical transmission approaches the analytical plane-wave value at its carrier energy $E_0$.

<details><summary><b>Final check: why is $P_{left}$ not equal to the reflection coefficient during the collision?</b></summary>

Before and during the collision, $P_{left}$ includes incident probability that has not yet reached the potential as well as reflected probability. Only after the reflected and transmitted packets are spatially separated can the late-time left and right integrals be interpreted approximately as $R$ and $T$.
</details>

## 6. Optional investigations

### A. Locate the threshold

For the step, approach $E/V_0=1$ from below and above. Explain the penetration below threshold and the very long wavelength and low velocity just above it.

### B. Measure exponential tunnelling

For fixed $E/V_0<1$, record the numerical transmission for several barrier widths. Plot $\ln T$ against width. In the regime where tunnelling is weak, is it approximately linear?

### C. Search for a transmission resonance

Choose $E/V_0>1$, vary the width, and locate a high-transmission case. Compare it with the condition $qa\approx n\pi$.

### D. Test the plane-wave limit

Increase $\sigma_x$. Does the numerical transmission move closer to the analytical value? Explain the result using $\Delta k\sim1/(2\sigma_x)$.

## 7. AI-assisted learning check — *Comprobación asistida por IAG*

This final activity uses generative AI as a **verification tool**, not as a substitute for your own reasoning.  
**Always answer first on your own.** Then use an AI assistant to identify possible errors, missing ideas or unclear reasoning, and compare its feedback with the notebook.

### 1. Write a ~200-word summary

In **about 200 words**, summarise what you have learned in this notebook. Your summary should include, in your own words:

- how a Gaussian wave packet differs from a single stationary plane wave;
- why reflection can occur at a potential step even when $E>V_0$;
- what happens at a step when $E<V_0$;
- why a **finite** barrier can produce tunnelling for $E<V_0$;
- how barrier width and energy affect transmission;
- why the final wave-packet probabilities can be compared with stationary $R$ and $T$ only after the packets separate.

**Only after writing your summary**, paste it into a generative-AI tool and use the following prompt:

> *Act as a critical tutor in undergraduate quantum mechanics. Read my summary but do not rewrite it immediately. First: (1) identify any conceptual errors, (2) point out important ideas that are missing, and (3) identify any statements that are ambiguous or imprecise. Explain each issue briefly. Base your feedback on standard quantum mechanics and do not invent results that are not supported by my explanation.*

Revise your summary only after considering the feedback critically.

---

### 2. Self-explanation

Without using AI first, explain:

> **Why can a particle with mean energy below a finite barrier appear on the far side without gaining energy, while a monochromatic state below an infinitely extended step has no propagating transmitted wave?**

Your explanation should explicitly discuss the **evanescent wave**, the **finite width of the barrier**, the **second boundary**, and **energy conservation**.

Then ask the AI:

> *Check my explanation step by step. Do not simply give me the correct answer. Tell me which step of my reasoning is correct, incomplete or incorrect, and ask me one follow-up question that would help me improve the explanation.*

---

### 3. Two verification questions

Answer both questions **before** consulting the AI.

**Q1.** For $E<V_0$, why does increasing the width of a finite rectangular barrier strongly reduce the transmission probability? Relate your explanation to the approximate dependence
$$
T\propto e^{-2\kappa a}.
$$

**Q2.** During the collision with a step or barrier, why is the probability $P_{\rm left}$ **not** equal to the reflection coefficient $R$? At what stage of the simulation can the left- and right-side probabilities be interpreted approximately as $R$ and $T$?

After answering, ask the AI:

> *Evaluate my answers to Q1 and Q2. For each one, state whether the reasoning is physically consistent and identify the first point that should be corrected, if any. Do not replace my answer with a model solution unless I explicitly ask for it.*

---

### 4. Final verification checklist

Before accepting the AI feedback, check:

- [ ] Does the explanation agree with the animations and numerical probabilities in this notebook?
- [ ] Is total energy conserved throughout the scattering process?
- [ ] Is the distinction between an evanescent tail and a propagating transmitted wave clear?
- [ ] Are $R$ and $T$ being interpreted only after incident, reflected and transmitted packets are spatially separated?
- [ ] If the AI disagrees with your result, can you verify the point using the equations, plots or simulations above?

> **Remember:** generative AI can produce plausible but incorrect explanations. The numerical simulation, the Schrödinger equation and your own physical reasoning remain the reference.
